# Week 35, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w035d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def cheapest(prices):
    return min(prices)

def dearest(prices):
    return max(prices)

def pick(prices, rule=cheapest):
    return rule(prices)

print(pick([450, 900, 120]), pick([450, 900, 120], dearest), pick([3, 1], lambda p: sum(p)))

**Predict before you run:** what will this print?



In [ ]:
handlers = []
def emit(x):
    return [h(x) for h in list(handlers)]

handlers.append(lambda x: x + 1)
first = emit(1)
handlers.append(lambda x: x * 10)
print(first, emit(2))

**Predict before you run:** what will this print?



In [ ]:
class Shape:
    kinds = {}
    def __init_subclass__(cls, kind, **kw):
        super().__init_subclass__(**kw)
        Shape.kinds[kind] = cls

class Dot(Shape, kind="dot"):
    pass

class Line(Shape, kind="line"):
    pass

print(sorted(Shape.kinds), type(Shape.kinds["dot"]()).__name__)

**Predict before you run:** what will this print?



In [ ]:
from functools import partial

def send(channel, to, text):
    return f"{channel}>{to}: {text}"

to_ada = partial(send, "sms", "Ada")
print(to_ada("hi"), to_ada(text="bye"))

**Predict before you run:** what will this print?



In [ ]:
class Base:
    def get(self):
        return "x"

class Wrap:
    def __init__(self, inner, mark):
        self.inner, self.mark = inner, mark
    def get(self):
        return self.mark + self.inner.get() + self.mark

print(Wrap(Wrap(Base(), "("), "*").get())

### Exercise: Strategies by name

Write three discount strategies as functions from a price in pence to a new price: `none` (unchanged), `student` (10 per cent off, rounded down) and `half` (half, rounded down). Put them in a dictionary `DISCOUNTS`, and write `charge(pence, name="none")`, which chooses by name in any case and raises `ValueError` listing the choices for an unknown name.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_discounts():\n    \"strategies\"\n    assert charge(1000) == 1000 and charge(1000, \"Student\") == 900 and charge(999, \"HALF\") == 499 and charge(5, \"student\") == 5\n    try:\n        charge(100, \"staff\")\n        assert False, \"an unknown name should raise ValueError\"\n    except ValueError as err:\n        assert \"half\" in str(err) and \"student\" in str(err)\n    DISCOUNTS[\"free\"] = lambda p: 0\n    assert charge(500, \"free\") == 0")

### Exercise: Watching a value

Write `Watched(value)`, a subject with a property `value`; setting a different value calls every subscriber as `handler(old, new)`. `subscribe(handler)` returns an unsubscribe function. Then write `Alarm(limit)`, a callable observer that records in `triggered` each new value that **crosses** above `limit` (the old value at or below it, the new one above).

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_watch():\n    \"observing\"\n    W, A = ns[\"Watched\"], ns[\"Alarm\"]\n    t = W(20)\n    alarm = A(25)\n    stop = t.subscribe(alarm)\n    for v in [24, 26, 27, 25, 30, 30]:\n        t.value = v\n    stop()\n    t.value = 20\n    t.value = 40\n    assert alarm.triggered == [26, 30] and t.value == 40\n    seen = []\n    t.subscribe(lambda o, n: seen.append((o, n)))\n    t.value = 40\n    t.value = 41\n    assert seen == [(40, 41)]")

### Exercise: Forge: a smart-home hub

Write the devices, factory, adapter, decorator, hub and strategies to the specification above.

In [ ]:
class AcmePlug:
    """Another maker's smart plug. Don't change it."""
    def __init__(self, label):
        self.label = label
        self._live = False

    def switch(self, on):
        self._live = bool(on)

    def is_live(self):
        return self._live

class Device:
    kinds = {}
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def setup():\n    H, L, Lg, P, A = ns[\"Hub\"], ns[\"Lamp\"], ns[\"Logged\"], ns[\"PlugAdapter\"], ns[\"AcmePlug\"]\n    hub = H()\n    log = []\n    hub.add(Lg(make_device({\"kind\": \"lamp\", \"name\": \"hall\"}), log))\n    hub.add(make_device({\"kind\": \"heater\", \"name\": \"lounge\"}))\n    hub.add(make_device({\"kind\": \"heater\", \"name\": \"bedroom\"}))\n    hub.add(Lg(P(A(\"kettle\")), log))\n    return hub, log\ndef test_devices():\n    \"devices and the factory\"\n    D, L, He = ns[\"Device\"], ns[\"Lamp\"], ns[\"Heater\"]\n    assert set(D.kinds) >= {\"lamp\", \"heater\"} and type(make_device({\"kind\": \"heater\", \"name\": \"x\"})) is He\n    lamp = L(\"hall\")\n    s = lamp.state()\n    s[\"on\"] = True\n    assert lamp.state() == {\"on\": False, \"brightness\": 100}, \"state() should return a new dictionary\"\n    lamp.command(\"dim\", level=40)\n    lamp.command(\"on\")\n    assert lamp.state() == {\"on\": True, \"brightness\": 40}\n    for f in [lambda: lamp.command(\"dim\", level=101), lambda: lamp.command(\"explode\"), lambda: He(\"h\").command(\"set\", target=40),\n              lambda: make_device({\"kind\": \"toaster\", \"name\": \"t\"}), lambda: make_device({\"kind\": \"lamp\", \"name\": \"x\", \"colour\": \"red\"})]:\n        try:\n            f()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\ndef test_adapter_and_log():\n    \"the adapter and the decorator\"\n    P, A, Lg, L = ns[\"PlugAdapter\"], ns[\"AcmePlug\"], ns[\"Logged\"], ns[\"Lamp\"]\n    plug = A(\"kettle\")\n    p = P(plug)\n    p.command(\"on\")\n    assert plug.is_live() and p.state() == {\"on\": True} and p.name == \"kettle\" and not issubclass(P, A)\n    log = []\n    lg = Lg(L(\"hall\"), log)\n    lg.command(\"dim\", level=40)\n    lg.command(\"on\")\n    try:\n        lg.command(\"dim\", level=-1)\n    except ValueError:\n        pass\n    assert log == [\"hall: dim level=40\", \"hall: on\"] and lg.name == \"hall\" and lg.state()[\"brightness\"] == 40\ndef test_hub():\n    \"the hub\"\n    hub, log = setup()\n    seen = []\n    stop = hub.subscribe(lambda name, action, state: seen.append((name, action, state)))\n    hub.command(\"hall\", \"on\")\n    hub.command(\"kettle\", \"on\")\n    stop()\n    hub.command(\"hall\", \"off\")\n    assert seen == [(\"hall\", \"on\", {\"on\": True, \"brightness\": 100}), (\"kettle\", \"on\", {\"on\": True})]\n    assert log == [\"hall: on\", \"kettle: on\", \"hall: off\"]\n    for f, err in [(lambda: hub.command(\"garage\", \"on\"), KeyError), (lambda: hub.add(ns[\"Lamp\"](\"hall\")), ValueError)]:\n        try:\n            f()\n            assert False, f\"should raise {err.__name__}\"\n        except err:\n            pass\ndef test_heating():\n    \"strategies\"\n    hub, log = setup()\n    sched = schedule({7: 21, 9: 16, 17: 21, 22: None})\n    assert [sched(h) for h in [0, 6, 7, 8, 9, 16, 17, 21, 22, 23]] == [None, None, 21, 21, 16, 16, 21, 21, None, None]\n    assert constant(19)(3) == 19\n    seen = []\n    hub.subscribe(lambda name, action, state: seen.append((name, action)))\n    assert hub.tick(7, sched) == 4 and seen == [(\"bedroom\", \"set\"), (\"bedroom\", \"on\"), (\"lounge\", \"set\"), (\"lounge\", \"on\")]\n    assert hub.device(\"lounge\").state() == {\"on\": True, \"target\": 21}\n    assert hub.tick(8, sched) == 0 and hub.tick(9, sched) == 2 and hub.tick(22, sched) == 2 and hub.tick(23, sched) == 0\n    assert hub.device(\"bedroom\").state()[\"on\"] is False and hub.tick(12, constant(18)) == 4")

> **Patterns as a vocabulary.** Count the patterns in the hub: a factory makes devices, an adapter fits the plug, a decorator logs commands, the hub is an observable subject, and heating schedules are strategies. Yet no single part is complicated, and each answers one question: how is it made, how does it fit, what extra does it do, who hears about it, how does it decide? That's the real value of patterns: not clever code, but a shared vocabulary for small, familiar shapes, so that someone reading `Logged(PlugAdapter(AcmePlug("kettle")), log)` understands it at a glance. Notice, too, what isn't here: no pattern was used where an `if` or a plain function would do.

### Exercise: Forge bonus: newsroom alerts

A newsroom sends stories to readers through channels, each reader choosing what they want. Build it from this week's patterns:

- `Story`, a frozen dataclass: `headline`, `section` and `urgency` (1 to 3, 3 being breaking news).
- **Filters** (strategies) are functions from a `Story` to `bool`: `section_is(name)`, `urgent_at_least(level)`, and `all_of(*filters)`, which is True only when every filter is (and True for no filters).
- **Channels** have `send(text)`. `Inbox` keeps every text in `messages`. Decorators: `Prefixed(inner, prefix)` adds the prefix; `Digest(inner, size)` collects texts and, every time it has `size` of them, sends **one** text joining them with `" | "`; `Capitals(inner)` sends the text in capitals.
- `make_channel(spec)`, a **recursive** factory: `{"type": "inbox"}`, `{"type": "prefixed", "prefix": "[NEWS] ", "inner": {...}}`, `{"type": "digest", "size": 2, "inner": {...}}`, `{"type": "capitals", "inner": {...}}`. An unknown type raises `ValueError`. It must return the `Inbox` it made as well, so the result is a pair `(channel, inbox)`.
- `Desk` (the subject): `subscribe(story_filter, channel)` returns an unsubscribe function; `publish(story)` sends the headline to every subscribed channel whose filter accepts the story, and returns how many channels it was sent to.

In [ ]:
from dataclasses import dataclass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parts():\n    \"filters and channels\"\n    S, I, P, D, C = (ns[n] for n in [\"Story\", \"Inbox\", \"Prefixed\", \"Digest\", \"Capitals\"])\n    st = S(\"Rail strike called off\", \"transport\", 2)\n    assert section_is(\"transport\")(st) and not urgent_at_least(3)(st) and all_of(section_is(\"transport\"), urgent_at_least(2))(st) and all_of()(st)\n    box = I()\n    d = D(C(box), 2)\n    for t in [\"one\", \"two\", \"three\"]:\n        d.send(t)\n    assert box.messages == [\"ONE | TWO\"]\n    box2 = I()\n    P(box2, \"[NEWS] \").send(\"x\")\n    assert box2.messages == [\"[NEWS] x\"]\ndef test_factory():\n    \"a recursive factory\"\n    ch, box = make_channel({\"type\": \"prefixed\", \"prefix\": \"[BREAKING] \", \"inner\": {\"type\": \"capitals\", \"inner\": {\"type\": \"inbox\"}}})\n    ch.send(\"Storm warning\")\n    assert box.messages == [\"[BREAKING] STORM WARNING\"]\n    ch2, box2 = make_channel({\"type\": \"capitals\", \"inner\": {\"type\": \"prefixed\", \"prefix\": \"[b] \", \"inner\": {\"type\": \"inbox\"}}})\n    ch2.send(\"x\")\n    assert box2.messages == [\"[b] X\"], \"capitals is the outer layer here, so the prefix is added after the text is capitalised\"\n    for bad in [{\"type\": \"pigeon\"}, {\"type\": \"digest\", \"size\": 2}]:\n        try:\n            make_channel(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except (ValueError, KeyError):\n            pass\ndef test_desk():\n    \"the desk\"\n    S, De = ns[\"Story\"], ns[\"Desk\"]\n    desk = De()\n    breaking, box1 = make_channel({\"type\": \"prefixed\", \"prefix\": \"[BREAKING] \", \"inner\": {\"type\": \"inbox\"}})\n    sport, box2 = make_channel({\"type\": \"digest\", \"size\": 2, \"inner\": {\"type\": \"inbox\"}})\n    desk.subscribe(urgent_at_least(3), breaking)\n    stop = desk.subscribe(section_is(\"sport\"), sport)\n    stories = [S(\"Eagles win\", \"sport\", 2), S(\"Storm warning\", \"weather\", 3), S(\"Derby tonight\", \"sport\", 1), S(\"Cup final moved\", \"sport\", 3)]\n    counts = [desk.publish(s) for s in stories]\n    assert counts == [1, 1, 1, 2] and box1.messages == [\"[BREAKING] Storm warning\", \"[BREAKING] Cup final moved\"]\n    assert box2.messages == [\"Eagles win | Derby tonight\"]\n    stop()\n    assert desk.publish(S(\"Transfer news\", \"sport\", 1)) == 0")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [Strategy](w035d1.html), [Observer](w035d2.html), [Factory](w035d3.html), [Adapter](w035d4.html) or [Decorator](w035d5.html).

## Mixed review

## Forge: a smart-home hub

A home hub controls lamps, heaters and plugs, and lets other parts of the program (a phone app, an energy report, a log) react to every change. Here's the specification, with the pattern each part uses.

**Devices, made by a factory.**

- `Device` is the base class. Subclasses register themselves with `__init_subclass__` and a `kind` keyword (`class Lamp(Device, kind="lamp")`) in `Device.kinds`. Every device has a `name`, a method `state()` returning a **new** dictionary describing it, and `command(action, **settings)`, which raises `ValueError` for an action it doesn't know.
- `Lamp(name)`: state `{"on": False, "brightness": 100}` to start. Actions `"on"`, `"off"`, and `"dim"` with a setting `level` (a whole number from 0 to 100, otherwise `ValueError`), which sets the brightness.
- `Heater(name)`: state `{"on": False, "target": 18}`. Actions `"on"`, `"off"`, and `"set"` with a setting `target` (5 to 30, otherwise `ValueError`).
- `make_device(spec)` makes a device from a dictionary such as `{"kind": "lamp", "name": "hall"}`; an unknown kind, or settings the class doesn't take, raise `ValueError`.

**A plug from another maker, adapted.** `AcmePlug` (given, and not to be changed) has `label`, `switch(on)` and `is_live()`. `PlugAdapter(plug)` makes it work like a device: its `name` is the plug's label, its state is `{"on": ...}` from `is_live()`, and it understands `"on"` and `"off"` (anything else raises `ValueError`). It must not inherit from `AcmePlug`.

**Logging, as a decorator.** `Logged(device, log)` wraps any device, with the same `name`, `state()` and `command(...)`. After a command succeeds, it appends a line to `log` like `hall: dim level=40` (the device's name, the action, then any settings as `key=value` in alphabetical order of key, all separated by single spaces). Failed commands aren't logged.

**The hub, as a subject.** `Hub()` holds devices by name:

- `add(device)` (a duplicate name raises `ValueError`) and `device(name)` (an unknown name raises `KeyError`);
- `subscribe(handler)` returns an unsubscribe function; after every successful command, each handler is called as `handler(name, action, state)` with the device's new state;
- `command(name, action, **settings)` passes the command to the device, then notifies.

**Heating, by strategy.** A heating strategy is a function from an hour (0 to 23) to a target temperature, or `None` meaning "off".

- `constant(temp)` returns a strategy that always gives `temp`.
- `schedule(times)` returns a strategy from a dictionary like `{7: 21, 9: 16, 17: 21, 22: None}`: at a given hour, use the entry for the latest hour at or before it; before the first entry of the day, use the **last** entry (from the night before).
- `Hub.tick(hour, strategy)` applies the strategy to every device whose state has a `"target"` key, in name order, using `command` so observers hear about it: for `None`, send `"off"` if it's on; for a temperature, send `"set"` with that target if it differs, and `"on"` if it's off. It returns the number of commands sent.

## Forge bonus: newsroom alerts

## Weekly quiz

## Week 35 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain what design patterns are, and use strategies as functions, callable objects and classes, chosen by name. ([Day 1](w035d1.html))
- I can write subjects and observers, event buses with named events, and handle the pitfalls of notification. ([Day 2](w035d2.html))
- I can write factory functions, alternative constructors and self-registering classes, and recognise Abstract Factory. ([Day 3](w035d3.html))
- I can write object, function and data adapters, use `functools.partial`, and build a facade. ([Day 4](w035d4.html))
- I can write object decorators and function wrappers, stack them in the right order, and tell them from adapters and proxies. ([Day 5](w035d5.html))
- I built a smart-home hub and a newsroom alert system using all five. (Today)

Tomorrow is a **rest day**. Next week: **more patterns and architecture**: State, Command and undo, Iterator, layered architecture and dependency injection.